# Section 01.01 — Python Generators and Lazy Iteration

**Data Science and Machine Learning Course** — Extra Section

---

## Learning Objectives

By the end of this section you will be able to:
- Create generator functions using `yield`
- Explain the difference between lazy and eager evaluation
- Advance generators manually with `next()`
- Build both finite and infinite generators
- Safely consume infinite generators with `itertools.islice`
- Measure and compare the memory footprint of lists vs generators
- Recognise where this pattern appears in data science and deep learning pipelines

---

## Part 1 — What Is a Generator?

A **generator** is a special Python function that produces values **one at a time**, pausing between each.

Instead of computing and returning all results at once (like a list), a generator uses the keyword `yield` to pause execution and hand one value to the caller. On the next call it resumes exactly where it left off.

This makes generators:
- **memory-efficient** — only one value exists in memory at a time
- **lazy** — values are computed only when requested
- **composable** — generators can feed into other generators

Real-world appearances of this pattern:

| Domain | Example |
|--------|---------|
| Data science | Reading large CSV files line by line |
| Deep learning | PyTorch `DataLoader` mini-batch iteration |
| NLP / LLMs | Token streaming in chat responses |
| Reinforcement learning | Environment step loops |
| Real-time systems | Sensor feeds, trading streams |

---

## Part 2 — Your First Generator

In [ ]:
def count_up_to(n):
    for i in range(n):
        yield i          # pauses here, hands i to caller, resumes on next call

for num in count_up_to(5):
    print(num)

The loop calls `count_up_to(5)` as an iterator. Each iteration triggers one `yield`, producing the next value.

---

## Part 3 — `yield` vs `return`

Understanding the difference is the core insight of this section.

In [ ]:
# A normal function: runs to completion, returns one value, done
def normal_function():
    return 5

print(normal_function())   # 5 — function exits

In [ ]:
# A generator function: pauses at yield, resumes on next call
def generator_function():
    yield 1
    yield 2
    yield 3

for val in generator_function():
    print(val)

> **Key idea:** `return` ends a function. `yield` **pauses** it — the local variables and execution position are preserved between calls. This pause-resume behaviour is **lazy evaluation**.

---

## Part 4 — Finite Fibonacci Generator

Fibonacci is a classic generator example: each value depends only on the previous two, making it ideal for sequential, stateful computation.

In [ ]:
def fibonacci_generator(n):
    """
    Yield the first n Fibonacci numbers.
    """
    if n < 0:
        raise ValueError("n must be non-negative")

    a, b = 0, 1
    for _ in range(n):
        yield a
        a, b = b, a + b

for num in fibonacci_generator(10):
    print(num, end=" ")

**Why this is better than a list:**

| Approach | Memory usage | Computed |
|----------|-------------|----------|
| `fib_list = [0,1,1,2,3,5,...]` | Stores all N values | Eagerly — all at once |
| `fibonacci_generator(N)` | Stores only current `a, b` | Lazily — one at a time |

For large N or streaming data, the generator wins decisively.

---

## Part 5 — Infinite Generator

A generator can run forever — the caller controls when to stop.

In [ ]:
def fibonacci_infinite():
    """Generate Fibonacci numbers indefinitely."""
    a, b = 0, 1
    while True:      # runs forever — the caller stops it
        yield a
        a, b = b, a + b

> **Never call `list()` on an infinite generator** — it will run until your system runs out of memory. Always use a stopping mechanism such as `next()` or `itertools.islice`.

---

## Part 6 — Manual Control with `next()`

`next()` pulls one value from the generator at a time — the clearest demonstration of lazy iteration.

In [ ]:
gen = fibonacci_infinite()   # create the generator object

print(next(gen))   # pulls value 1
print(next(gen))   # pulls value 2
print(next(gen))   # pulls value 3
print(next(gen))   # pulls value 4
print(next(gen))   # pulls value 5

Each `next()` call resumes the generator from exactly where it last paused.

> **`StopIteration`:** when a finite generator is exhausted and you call `next()` once more, Python raises `StopIteration`. A `for` loop catches this automatically — that is how it knows when to stop.

In [ ]:
# Demonstrating StopIteration on a finite generator
small_gen = fibonacci_generator(3)

print(next(small_gen))   # 0
print(next(small_gen))   # 1
print(next(small_gen))   # 1

try:
    print(next(small_gen))   # generator is exhausted
except StopIteration:
    print("Generator exhausted — StopIteration raised")

---

## Part 7 — Safe Consumption with `itertools.islice`

`islice` slices a generator lazily — the same way list slicing works, but without materialising the full sequence.

In [ ]:
from itertools import islice

# Take the first 10 values
result = list(islice(fibonacci_infinite(), 10))
print(result)

In [ ]:
# islice(iterable, start, stop) — values at index 10 through 19
result = list(islice(fibonacci_infinite(), 10, 20))
print(result)

In [ ]:
# islice(iterable, start, stop, step) — every second value from index 0 to 19
result = list(islice(fibonacci_infinite(), 0, 20, 2))
print(result)

**`islice` syntax mirrors list slicing:**

| `islice` | Equivalent list slice |
|----------|-----------------------|
| `islice(gen, stop)` | `lst[:stop]` |
| `islice(gen, start, stop)` | `lst[start:stop]` |
| `islice(gen, start, stop, step)` | `lst[start:stop:step]` |

The critical difference: `islice` never materialises the full sequence.

---

## Part 8 — Memory Comparison: List vs Generator

In [ ]:
import sys

N = 10_000

# List comprehension: all N integers stored in memory
num_list = [x for x in range(N)]

# Generator expression: only the iterator state is stored
num_gen  = (x for x in range(N))

print(f"List    ({N:,} elements): {sys.getsizeof(num_list):,} bytes")
print(f"Generator (same range):   {sys.getsizeof(num_gen):,} bytes")

The generator size stays constant regardless of N — it holds only the iterator state, not the values.

> **Complexity:**
> - **Time:** O(n) — each value is computed once
> - **Space:** O(1) — only the current state is stored

---

## Part 9 — Generator Expressions

Just as list comprehensions use `[]`, generator expressions use `()`. They create a generator inline without a `def`.

In [ ]:
# List comprehension — eager, all values stored
squares_list = [x**2 for x in range(10)]
print("List:", squares_list)

# Generator expression — lazy, nothing computed yet
squares_gen = (x**2 for x in range(10))
print("Generator:", squares_gen)          # shows object, not values
print("First value:", next(squares_gen))  # compute on demand

In [ ]:
# Generator expressions work directly in function calls — no extra parentheses needed
total = sum(x**2 for x in range(1000))
print(f"Sum of first 1000 squares: {total:,}")

---

## Part 10 — Real Data Science Pattern

The most common generator in data science is file reading — processing large datasets line by line without loading them fully into memory.

In [ ]:
def read_large_file(file_path):
    """
    Yield one line at a time from a large text file.
    Never loads the full file into memory.
    """
    with open(file_path, 'r') as f:
        for line in f:
            yield line.strip()

# Usage pattern:
# for line in read_large_file('huge_dataset.csv'):
#     process(line)

print("Pattern defined — works for files of any size.")

This pattern scales to:
- Multi-gigabyte CSV files
- Log processing and ETL pipelines
- NLP corpora (one sentence at a time)
- Large text datasets

---

## Part 11 — Deep Learning Connection

Later in this course, this exact concept appears under different names:

In [ ]:
# Mini-batch generator — the conceptual core of PyTorch DataLoader
import numpy as np

def mini_batch_generator(data, batch_size):
    """
    Yield consecutive batches of batch_size from data.
    This is exactly how PyTorch DataLoader works under the hood.
    """
    for start in range(0, len(data), batch_size):
        yield data[start : start + batch_size]

dataset = np.arange(20)   # simulate 20 training samples

for batch_num, batch in enumerate(mini_batch_generator(dataset, batch_size=4)):
    print(f"Batch {batch_num}: {batch}")

**Where generators appear in ML/DL:**

| Tool | Generator role |
|------|---------------|
| `torch.utils.data.DataLoader` | Mini-batch iteration |
| Keras `fit_generator` | Streaming augmented data |
| LLM inference APIs | Token-by-token streaming |
| RL environments | `env.step()` loop state |
| Streaming ETL | Record-by-record processing |

---

## Part 12 — Mini Exercise

Attempt each before reading the solution below.

In [ ]:
# Exercise 1: Write a generator that yields even numbers from 0 up to (but not including) n
# Your code here


In [ ]:
# Solution 1
def even_numbers(n):
    for i in range(0, n, 2):
        yield i

print(list(even_numbers(20)))

In [ ]:
# Exercise 2: Use a generator expression to compute the sum of squares of all odd numbers < 100
# Your code here


In [ ]:
# Solution 2
result = sum(x**2 for x in range(1, 100, 2))
print(result)

In [ ]:
# Exercise 3: Use islice to print only the 5th through 10th Fibonacci numbers (index 4 to 9)
# Your code here


In [ ]:
# Solution 3
from itertools import islice

result = list(islice(fibonacci_infinite(), 4, 10))
print(result)

---

## Part 13 — When to Use (and Avoid) Generators

**Use generators when:**
- The dataset is large or the full size is unknown
- Values are consumed sequentially (one pass)
- Memory is constrained
- The sequence may be infinite
- Building streaming or pipeline architectures

**Avoid generators when:**
- You need random access by index (`gen[5]` does not work)
- You need to iterate over the data multiple times (a generator is exhausted after one pass)
- The full sequence needs to be materialised anyway

> **Interview answer:** "Why use a generator instead of a list?" → Generators provide lazy evaluation, O(1) memory regardless of N, support for infinite sequences, and natural composability for data pipelines.

---

## Key Takeaways

- `yield` pauses a function and returns one value — the function resumes from that point on the next call.
- Generators are **lazy**: values are computed only when requested, not all at once.
- Memory footprint is **O(1)** — only the current state is stored, not all N values.
- `next()` manually pulls one value; a `for` loop calls `next()` automatically until `StopIteration`.
- Infinite generators are powerful — always pair them with `itertools.islice` or a loop with a break condition.
- **Generator expressions** `(expr for item in iterable)` are the inline, no-`def` form.
- This pattern is the conceptual foundation of PyTorch `DataLoader`, LLM token streaming, and all lazy data pipelines.

---

*This notebook supplements **Section 01 — Course Introduction**. Continue to **Section 02 — Environment Setup**.*